In [0]:
%sql
DROP CONNECTION IF EXISTS dev_earthquake_conn;

CREATE CONNECTION dev_earthquake_conn
TYPE HTTP
OPTIONS (
    host 'https://earthquake.usgs.gov',
    port '443',
    base_path '/earthquakes/feed/v1.0',
    bearer_token 'na'
)

In [0]:
from databricks.sdk import WorkspaceClient 
w = WorkspaceClient()
conn = w.connections.get('dev_earthquake_conn')
base_url = f"{conn.options['host']}{conn.options['base_path']}"


In [0]:
dbutils.widgets.text('catalog_name',"earthquake")
catalog_name = dbutils.widgets.get('catalog_name')

In [0]:
spark.sql(f"use catalog {catalog_name}");
spark.sql("use schema bronze");
spark.sql("create volume if not exists earthquake_data")

In [0]:
%sql
use catalog earthquake;
use schema bronze;
create volume if not exists earthquake_data;


In [0]:
import requests
import json
import datetime

url = f"{base_url}/summary/all_day.geojson"
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"error in getting data from {url}")
response.raise_for_status()  # Raise error for 4xx/5xx status codes
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/earthquake/bronze/earthquake_data/earthquake_date_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)